# Notebook 6: Aiven Managed OpenSearch
**Move from local Docker to a fully managed cloud cluster**

So far we've used a local single-node OpenSearch cluster via Docker.
In this notebook we connect to [Aiven for OpenSearch](https://aiven.io/opensearch): a fully managed,
multi-cloud service with automatic backups, TLS, and built-in k-NN support.

### What you'll do
1. Connect to an Aiven OpenSearch cluster over TLS
2. Create the same k-NN index on the managed cluster
3. Index all podcast chunks (re-using embeddings from Notebook 01)
4. Run the same RAG queries against the cloud cluster

### Prerequisites
- An Aiven account (free trial: https://console.aiven.io/signup)
- An Aiven for OpenSearch service running
- Connection details: host, port, username (`avnadmin`), password
- Notebooks 01 completed (to compare results)

## 0 · Connection Details

Find these in the Aiven Console → your OpenSearch service → **Overview** tab → **Connection information**.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from dotenv import load_dotenv
load_dotenv()

# Aiven connection details
AIVEN_HOST     = os.getenv('AIVEN_OPENSEARCH_HOST')
AIVEN_PORT     = os.getenv('AIVEN_OPENSEARCH_PORT')
AIVEN_USER     = os.getenv('AIVEN_OPENSEARCH_USER')
AIVEN_PASSWORD = os.getenv('AIVEN_OPENSEARCH_PASSWORD')
# ─────────────────────────────────────────────────────────────────────────

print(f'Will connect to: {AIVEN_HOST}:{AIVEN_PORT}')

Will connect to: os-5c86f19-florgflorg.i.aivencloud.com:23471


## 1 · Connect to Aiven OpenSearch

Aiven clusters use **TLS** with password authentication.
The `get_aiven_client()` helper handles `use_ssl=True` and `verify_certs=True`.

In [2]:
from src.opensearch_client import get_aiven_client

aiven_client = get_aiven_client(
    host=AIVEN_HOST,
    port=AIVEN_PORT,
    username=AIVEN_USER,
    password=AIVEN_PASSWORD,
)

info = aiven_client.info()
print(f"✅ Connected to Aiven OpenSearch {info['version']['number']}")
print(f"   Cluster: {info['cluster_name']}")
print(f"   Plugins: {', '.join(p['name'] for p in info.get('plugins', []) if 'knn' in p['name'].lower()) or 'k-NN built-in'}")

✅ Connected to Aiven OpenSearch 2.19.5
   Cluster: c49f193b-a481-422c-9b5f-c8eb611e1d7b
   Plugins: k-NN built-in


## 2 · Create the k-NN Index on Aiven

Same index schema as our local cluster. Aiven's OpenSearch includes the k-NN plugin by default.

In [4]:
from src.opensearch_client import INDEX_SETTINGS

AIVEN_INDEX = 'vector-podcast'

exists = aiven_client.indices.exists(AIVEN_INDEX)
if exists:
    print(f"Index '{AIVEN_INDEX}' already exists on Aiven")
    count = aiven_client.count(index=AIVEN_INDEX)['count']
    print(f"  Documents: {count}")
else:
    aiven_client.indices.create(AIVEN_INDEX, body=INDEX_SETTINGS)
    print(f"✅ Created index '{AIVEN_INDEX}' on Aiven")

Index 'vector-podcast' already exists on Aiven
  Documents: 0


## 3 · Load Data & Embed

We re-use the same parser and embedding model from Notebook 01.

In [5]:
from sentence_transformers import SentenceTransformer
from src.parser import load_podcast_chunks

DATA_DIR = os.path.join(os.getcwd(), '..', 'vector-podcast')
chunks = load_podcast_chunks(DATA_DIR, chunk_size=400, overlap=50)
print(f'Loaded {len(chunks)} chunks from {len(set(c.episode_title for c in chunks))} episodes')

model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [c.chunk_text for c in chunks]

print(f'Embedding {len(texts)} chunks...')
embeddings = model.encode(
    texts, batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
print(f'✅ Embeddings: {embeddings.shape}')

Loaded 1018 chunks from 33 episodes
Embedding 1018 chunks...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Embeddings: (1018, 384)


## 4 · Bulk-Index into Aiven

Indexing to a remote cluster is slower than localhost, bulk batching matters more.

In [6]:
from src.opensearch_client import bulk_index
import time

embedding_lists = embeddings.tolist()

print('Indexing into Aiven OpenSearch (this may take a minute over the network)...')
t0 = time.time()

# Temporarily patch INDEX_NAME for bulk_index to target the aiven index
import src.opensearch_client as osc
orig_name = osc.INDEX_NAME
osc.INDEX_NAME = AIVEN_INDEX

indexed = bulk_index(aiven_client, chunks, embedding_lists, batch_size=200)

osc.INDEX_NAME = orig_name  # restore

elapsed = time.time() - t0
print(f'\n✅ Indexed {indexed} documents in {elapsed:.1f}s')

aiven_client.indices.refresh(AIVEN_INDEX)
count = aiven_client.count(index=AIVEN_INDEX)['count']
print(f'   Documents in Aiven index: {count}')

Indexing into Aiven OpenSearch (this may take a minute over the network)...

✅ Indexed 1018 documents in 3.2s
   Documents in Aiven index: 1018


## 5 · Search the Aiven Cluster

In [7]:
def embed(text: str) -> list[float]:
    return model.encode(text, normalize_embeddings=True).tolist()


def search_aiven(query: str, k: int = 5):
    query_vec = embed(query)
    body = {
        'size': k,
        'query': {
            'knn': {
                'embedding': {'vector': query_vec, 'k': k}
            }
        }
    }
    resp = aiven_client.search(index=AIVEN_INDEX, body=body)
    return [
        {
            'score': hit['_score'],
            'episode_title': hit['_source']['episode_title'],
            'chunk_text': hit['_source']['chunk_text'],
            'url': hit['_source']['url'],
            'pub_date': hit['_source']['pub_date'],
            'chunk_index': hit['_source']['chunk_index'],
        }
        for hit in resp['hits']['hits']
    ]


query = 'What is HNSW and why is it so widely adopted?'
results = search_aiven(query, k=3)

print(f'Query: "{query}"')
print(f'Cluster: Aiven ({AIVEN_HOST})')
print('─' * 70)
for i, r in enumerate(results, 1):
    print(f'  [{i}] {r["score"]:.4f}  {r["episode_title"]}')
    print(f'       {r["chunk_text"][:150]}...')
    print()

Query: "What is HNSW and why is it so widely adopted?"
Cluster: Aiven (os-5c86f19-florgflorg.i.aivencloud.com)
──────────────────────────────────────────────────────────────────────
  [1] 0.6837  Joan Fontanals - Principal Engineer - Jina AI
       also python? I don't know if we are for instance I think we are also using some bindings for HLW so you are using probably C++ version of HNSW binding...

  [2] 0.6786  Jo Bergum - Distinguished Engineer, Yahoo! Vespa - Journey of Vespa from Sparse into Neural Search
       seen was also developing right quite quite fast. Did you kind of look at what that team is doing which is like an open source project? Was there somet...

  [3] 0.6673  Jo Bergum - Distinguished Engineer, Yahoo! Vespa - Journey of Vespa from Sparse into Neural Search
       then you can compute the overlap between those two and that's typically then what's used in the recall at k right so I did two blog posts on what I ca...



## 6 · RAG on Aiven

In [8]:
from src.rag import ask_streaming

query = 'What are wormhole vectors and how do they improve hybrid search?'
hits = search_aiven(query, k=5)

print(f'Question: {query}')
print(f'Sources:  {", ".join(set(h["episode_title"] for h in hits))}')
print('─' * 70)
ask_streaming(query, hits)

Question: What are wormhole vectors and how do they improve hybrid search?
Sources:  Adding ML layer to Search: Hybrid Search Optimizer with Daniel Wrigley and Eric Pugh, Trey Grainger - Wormhole Vectors
──────────────────────────────────────────────────────────────────────
Wormhole vectors are a concept used to translate a query from one vector space to find a corresponding region in a different vector space that shares the same semantic meaning (according to Trey Grainger - Wormhole Vectors).

The technique improves search by allowing traversal across different vector spaces:

*   **Mechanism:** The process involves querying the initial vector space to obtain a relevant document set. A wormhole vector is then derived from this document set.
*   **Traversal:** This generated vector can be used to query (or "hop" or "traverse") the second vector space, allowing the search to arrive at a region that semantically resembles the initial query's meaning.

Therefore, the general idea is to q

## Key Differences: Local vs Aiven

| Aspect | Local Docker | Aiven Managed |
|---|---|---|
| Setup | `docker compose up` | Click in console / Terraform |
| TLS | Disabled for dev | Always on, managed certs |
| Auth | None (security plugin off) | Username + password (or SSO) |
| Backups | Manual | Automatic, point-in-time |
| Scaling | Single node | Resize or add nodes |
| Multi-cloud | N/A | AWS, GCP, Azure |
| k-NN plugin | Included | Included by default |
| Dashboards | Separate container | Included with service |

### Aiven extras for production
- **Cross-cluster replication** for disaster recovery
- **Index state management** for automated rollover and retention
- **VPC peering** for private network access
- **Terraform provider** for infrastructure-as-code

Try it: https://console.aiven.io/signup